# ReAct Prompting

ReAct (Reason+Act) combines reasoning and acting LLM capabilities in the same model.

Models are the reasoning engine of agents. They drive the agent’s decision-making process, determining which tools to call, how to interpret results, and when to provide a final answer.

ReAct enables language models to generate both verbal reasoning traces and text actions in an interleaved manner.

While actions lead to observation feedback from an external environment (“Env” in the figure below), reasoning traces do not affect the external environment.

Instead, they affect the internal state of the model by reasoning over the context and updating it with useful information to support future reasoning and acting.

## Working with local models using Ollama

In [ ]:
import os 

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

model_name = "qwen3-vl:4b-instruct"
# model = 'qwen3-vl:2b-thinking'

ollama_service = "http://localhost:11434/v1/"
temperature = 0
message1 = "Hi"

### Using Ollama library

In [ ]:
from ollama import chat

In [ ]:
response = chat(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": message1,
        },
    ],
)

print(response.message.content)

### Using OpenAI API

using Chat Completions

In [ ]:
from openai import OpenAI

In [ ]:
client = OpenAI(base_url=ollama_service)

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": message1,
        }
    ],
    temperature=temperature,
)

print(response.choices[0].message.content)

using Responses API (newest API)

In [ ]:
client = OpenAI(base_url=ollama_service)

response = client.responses.create(
    model=model_name, input=message1, temperature=temperature
)

print(response.output_text)

### Using Langchain

In [ ]:
from langchain_core.messages import HumanMessage

Using [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama)

In [ ]:
from langchain_ollama import ChatOllama

In [ ]:
model = ChatOllama(model=model_name, temperature=temperature)

message = HumanMessage(content=message1)
response = model.invoke([message])

print(response.content)

Using [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI(base_url=ollama_service, model=model_name, temperature=temperature)

message = HumanMessage(content=message1)
response = model.invoke([message])

print(response.content)

**Model features**

| Feature | ChatOllama | ChatOpenAI |
| :--- | :---: | :---: |
| Tool calling | ✅ | ✅ |
| Structured output | ✅ | ✅ |
| Image input | ✅ | ✅ |
| Audio input | ❌ | ✅ |
| Video input | ❌ | ❌ |
| Token-level streaming | ✅ | ✅ |
| Native async | ✅ | ✅ |
| Token usage | ❌ | ✅ |
| Logprobs | ❌ | ✅ |

## Simple LLM graph (simple reasoning)

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from IPython.display import Image

There are different message types:,
- **SystemMessage** - Instructions for how the model should behave,
- **HumanMessage** - User input,
- **AIMessage** - Model responses,
- **ToolMessage** - Results from tool executions

In [ ]:
class LLM:
    def __init__(self, model_name=None, temperature=0):
        if not model_name:
            model_name = "qwen3-vl:4b-instruct"

        model = ChatOllama(model=model_name, temperature=temperature)

        self.model = model

    def __call__(self, state: MessagesState):
        return {"messages": [self.model.invoke(state["messages"])]}

In [ ]:
llm = LLM()

message = HumanMessage("Hi")
state = llm({"messages": [message]})
print(state)

In [ ]:
ai_message = state["messages"].pop()
print(f"AI: {ai_message.content}")

In [ ]:
ai_message.model_dump()

`StateGraph` is a graph whose nodes communicate by reading and writing to a shared state.

Nodes do the work, edges tell what to do next

In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("llm", LLM())

graph.add_edge(START, "llm")
graph.add_edge("llm", END)

llm = graph.compile()

Image(llm.get_graph().draw_mermaid_png())

In [ ]:
message = HumanMessage("Hi")
state = llm.invoke({"messages": [message]})
state

In [ ]:
def print_messages(state):
    for message in state["messages"]:
        message_dict = message.model_dump()

        type = message_dict["type"]
        content = message_dict["content"]

        print(f"{type.upper()}: {content}")

In [ ]:
print_messages(state)

In [ ]:
llm_reponse = state["messages"][-1].content
print(llm_reponse)

## LLM loop (conditional edges)

In [ ]:
def increase_message(state):
    messages = state["messages"]
    last_message = messages[-1]
    number = int(last_message.content)
    return {"messages": [HumanMessage(f"Increase the number by 1: {number}")]}

def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    return "5" if last_message.content == '5' else "<5"

In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("llm", LLM())
graph.add_node("increase_message", increase_message)

graph.add_edge(START, "increase_message")
graph.add_edge("increase_message", "llm")

graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        '5': END,
        '<5': "increase_message"
    }
)

llm = graph.compile()

Image(llm.get_graph().draw_mermaid_png())

In [ ]:
message = HumanMessage("0")
state = llm.invoke({"messages": [message]})

print_messages(state)

## Act (adding tools)

`FunctionGemma` is a model tuned for function calling.

In [ ]:
llm = LLM(model_name="functiongemma")

message = HumanMessage("Hi")
state = llm({"messages": [message]})

print_messages(state)

In [ ]:
class LLM:
    def __init__(self, model_name=None, temperature=0, tools=None):
        if not model_name:
            model_name = "qwen3-vl:4b-instruct"

        model = ChatOllama(model=model_name, temperature=temperature)

        ###################################
        if tools:
            model = model.bind_tools(tools)
        ###################################

        self.model = model

    def __call__(self, state: MessagesState):
        return {"messages": [self.model.invoke(state["messages"])]}

In [ ]:
from langchain_core.messages import ToolMessage
from langchain.tools import tool

The `@tool` decorator ensures that the function’s name, description, and parameter schema are all available to the LLM for tool calling.

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b

In [ ]:
add.name

In [ ]:
add.description

In [ ]:
add.args

Invoking the tool

In [ ]:
add_input = {"a": 2, "b": 3}

add.invoke(add_input)

Creating node to handle tool calls

In [ ]:

tools = [add]
tools_by_name = {tool.name: tool for tool in tools}


def tool_node(state: dict):
    messages = state["messages"]
    last_message = messages[-1]
    result = []

    for tool_call in last_message.tool_calls:
        print(f"Invoking tool: '{tool_call['name']}' with args: {tool_call['args']}")

        tool = tools_by_name[tool_call["name"]]
        args = tool_call["args"]

        observation = tool.invoke(args)
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))

    return {"messages": result}


def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return True

    return False

Creating an Act agent

In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("llm", LLM(model_name="functiongemma", tools=tools))
graph.add_node("tool", tool_node)

graph.add_edge(START, "llm")
graph.add_edge("tool", "llm")
graph.add_conditional_edges(
    "llm",
    should_continue,
    {
        True: 'tool',
        False: END
    }
)

act_agent = graph.compile()


Image(act_agent.get_graph().draw_mermaid_png())

When you register a tool with the LLM, you expose both the function signature (name and parameters) and a textual description

In [ ]:
message = HumanMessage("Add 3 and 4")

`functiongemma` model cannot reason

In [ ]:
llm = LLM(model_name="functiongemma")
state = llm({"messages": [message]})

print_messages(state)

`qwen3` model can reason

In [ ]:
llm = LLM()
state = llm({"messages": [message]})

print_messages(state)

Helping `functiongemma` with a tool to act

In [ ]:
state = act_agent.invoke({"messages": [message]})

The model decides, at runtime, when and how to use each tool based on the user's input and the context of the conversation

In [ ]:
print_messages(state)

A model is aware of the available tools

In [ ]:
message = HumanMessage("Increase 5")
state = act_agent.invoke({"messages": [message]})
print_messages(state)